# 5. Repensando el Overfitting

#### 5.1 Seteo del ambiente local


Esta parte se debe correr con un kernel de R local.
<br>En Jupyter, seleccionar el kernel **R** antes de ejecutar el notebook.


Los archivos persistentes quedan en el repo local: datasets en `datasets/` y resultados en `exp/`.


In [1]:
# Seteo local: resuelve el root del repo (funciona desde raiz, src/arboles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")


Para correr localmente, el dataset debe estar en `datasets/` dentro del repo.

<br>Si se va a subir a Kaggle, copiar `kaggle.json` a la raiz del repo antes de correr la siguiente celda. La celda lo instala en `~/.kaggle/kaggle.json` con permisos correctos.


In [2]:
dir.create(DATA_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)

dataset_local <- file.path(DATA_DIR, "dataset_pequeno.csv")
if (!file.exists(dataset_local)) {
  stop("No encuentro el dataset en: ", dataset_local)
}

if (file.exists(KAGGLE_JSON)) {
  kaggle_dir <- path.expand("~/.kaggle")
  dir.create(kaggle_dir, recursive = TRUE, showWarnings = FALSE)
  file.copy(KAGGLE_JSON, file.path(kaggle_dir, "kaggle.json"), overwrite = TRUE)
  Sys.chmod(file.path(kaggle_dir, "kaggle.json"), mode = "0600")
} else {
  message("No encontre kaggle.json en la raiz del repo. Solo es necesario para hacer submit a Kaggle.")
}




---



## 5.2 rpart  Canaritos

Se agregarán canaritos al dataset, se entrenará el arbol con los mejores hiperparámetros encontrados, y se analizará si los canaritos aparecen en algun split.

Esta parte se debe correr con el kernel de **R**.


limpio el ambiente de R

In [3]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,673608,36.0,1489770,79.6,NA,1489770,79.6
Vcells,1252763,9.6,8388608,64.0,49152,2014429,15.4


In [4]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart

Loading required package: rpart.plot



### 5.2.1  carga manual de hiperparámetros
Aqui debe cargar SU semilla primigenia y
<br> SUS mejores hiperparámetros que encontró para el ARBOL DE DECISION, ya sea por Grid Search o  Bayesian Optimization

In [5]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$rpart$cp <- -0.5
PARAM$rpart$minsplit <- 600
PARAM$rpart$minbucket <- 150
PARAM$rpart$maxdepth <- 6

In [6]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
# Seteo local: resuelve el root del repo (funciona desde raiz, src/arboles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento <- "exp5200"
dir.create(file.path(EXP_DIR, experimento), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento))


In [7]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))


In [8]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [12]:
PARAM$semila_primigenia

NULL

In [9]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

la siguiente celda tarda 4 minutos en correr

In [10]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [11]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

pdf 
  2

Los resultados quedan en el repo local: **`exp/exp5200/`**
<br> abra el archivo **arbol_canaritos.pdf**
<br> abra el .pdf con un lector de PDF
<br> y dentro del .pdf busque splits hechos en alguna de las nuevas variables canaritos




---



## 5.3 rpart  Canaritos desconfiados

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [13]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,813996,43.5,1489770,79.6,NA,1489770,79.6
Vcells,1598805,12.2,172230838,1314.1,49152,215288104,1642.6


In [14]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia y
<br> parametros de un@ alumn@ desconfiad@

In [15]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$rpart$cp <- -0.5
PARAM$rpart$minsplit <- 2000
PARAM$rpart$minbucket <- 800
PARAM$rpart$maxdepth <- 6

In [16]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
# Seteo local: resuelve el root del repo (funciona desde raiz, src/arboles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento <- "exp5300"
dir.create(file.path(EXP_DIR, experimento), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento))


In [ ]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))


In [ ]:
# me quedo solo con los datos de julio
dataset <- dataset[ foto_mes==202107,]

In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:154 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

la siguiente celda tarda 4 minutos en correr

In [ ]:
# Entreno el modelo

modelo <- rpart(formula= "clase_ternaria ~ .",
  data= dataset,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart
)


In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file = "arbol_canaritos_desconfiados.pdf", width=28, height=4)
prp(modelo, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

Los resultados quedan en el repo local: **`exp/exp5300/`**
<br> abra el archivo **arbol_canaritos_desconfiados.pdf**
<br> abra el .pdf con un lector de PDF
<br> y dentro del .pdf busque splits hecho en alguna de las nuevas variables canaritos




---



## 5.4 rpart  Canaritos pruning

Se trabaja con la original clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 1021917

PARAM$peso <- 10

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1
PARAM$rpart$maxdepth <- 16  # deberia pohner 31, por velocidad en la clase va el 16

In [ ]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
# Seteo local: resuelve el root del repo (funciona desde raiz, src/arboles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento <- "exp5400"
dir.create(file.path(EXP_DIR, experimento), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento))


In [ ]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))


In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:155 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [ ]:
# datos de training
dtrain <- dataset[foto_mes == 202107]

la siguiente celda corre en 12 minutos

In [ ]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_ternaria=="BAJA+2", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_ternaria ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [ ]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
    modelo_original$frame$var %like% "canarito",
    "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file = "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

In [ ]:
# datos del futuro
dfuture <- dataset[foto_mes == 202109]

In [ ]:
# scoring, aplico el modelo a los datos del futuro
prediccion <- predict(modelo_pruned,
  dfuture,
  type= "prob"
)

In [ ]:
# tabla prediccion
tb_prediccion <- as.data.table(list(
  "numero_de_cliente" = dfuture$numero_de_cliente,
  "prob"=prediccion[, "BAJA+2"]
))

In [ ]:
# Decison
PARAM$prob_corte <-  PARAM$peso/ ( PARAM$peso + 39)

tb_prediccion[ , Predicted := ifelse( prob< PARAM$prob_corte, 0L, 1L) ]

In [ ]:
# archivo para kaggle
archivo_kaggle <- "K5400_001.csv"

fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

In [ ]:
# subida a Kaggle
comando <- "kaggle competitions submit"
competencia <- "-c data-mining-inicial-2026-b"
arch <- paste( "-f", archivo_kaggle)

In [ ]:
mensaje <- paste0("-m 'cp=", PARAM$rpart$cp,
  "  minsplit=", PARAM$rpart$minsplit,
  "  minbucket=", PARAM$rpart$minbucket,
  "  maxdepth=", PARAM$rpart$maxdepth,
  "  peso=", PARAM$peso,
  "'"
 )

In [ ]:
linea <- paste( comando, competencia, arch, mensaje)

# este es el comando que correria desde el prompt de Linux
linea

In [ ]:
# ejecuto el comando
salida <- system(linea, intern=TRUE)
cat(salida)

Los resultados quedan en el repo local: **`exp/exp5400/`**
<br> abra el archivo **stopping_at_canaritos.pdf**
<br> abra el .pdf con un lector de PDF




---



## 5.5 rpart  Canaritos pruning BINARIA

Pasamos a trabajar con una clase  Binaria


*   POS = { BAJA+1, BAJA+2 }
*   NEG = { CONTINUA }




ahora la probabilidad que devuelve el modelo es de POS,
<br> ya no es la de BAJA+2,
<br> ya no puedo cortar por ella
<br> debo cortar por cnatidad de envios !

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
if(!require("rpart.plot")) install.packages("rpart.plot")
require("rpart.plot")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 1021917

PARAM$envios <- 9000
PARAM$peso <- 10

# Dejo crecer el arbol sin ninguna limitacion
# sin limite de altura ( 30 es el maximo que permite rpart )
# sin limite de minsplit ( 2 es el minimo natural )
# sin limite de minbukcet( 1 es el minimo natural )
# ya aprendimos que cp debe ser negativo
PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 2
PARAM$rpart$minbucket <- 1
PARAM$rpart$maxdepth <- 16 # deberia ser 31, por velocidad en clsae se baja  16

In [ ]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
# Seteo local: resuelve el root del repo (funciona desde raiz, src/arboles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento <- "exp5500"
dir.create(file.path(EXP_DIR, experimento), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento))


In [ ]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))


In [ ]:
# uso esta semilla para los canaritos
set.seed(PARAM$semila_primigenia)

# agrego los siguientes canaritos
for( i in 1:155 ) dataset[ , paste0("canarito", i ) :=  runif( nrow(dataset)) ]

In [ ]:
# datos de training
dtrain <- dataset[foto_mes == 202107]

In [ ]:
# clase binaria
dtrain[, clase_binaria2 := ifelse( clase_ternaria=="CONTINUA", "NEG", "POS" ) ]
dtrain[, clase_ternaria := NULL ]

la siguiente celda corre en 11 minutos

In [ ]:
# Entreno el modelo
pesos <- dtrain[, ifelse( clase_binaria2=="POS", PARAM$peso, 1.0 ) ]

modelo_original <- rpart(formula= "clase_binaria2 ~ .",
  data= dtrain,
  model= TRUE,
  xval= 0,
  control= PARAM$rpart,
  weights= pesos
)


In [ ]:
# hago el pruning de los canaritos
# haciendo un hackeo a la estructura  modelo_original$frame
# -666 es un valor arbritrariamente negativo que jamas es generado por rpart

modelo_original$frame[
    modelo_original$frame$var %like% "canarito",
    "complexity"
] <- -666

modelo_pruned <- prune(modelo_original, -666)

In [ ]:
# genero un pdf con el dibujo del arbol

pdf(file = "stopping_at_canaritos.pdf", width=28, height=4)
prp(modelo_pruned, extra=101, digits=5, branch=1, type=4, varlen=0, faclen=0)
dev.off()

In [ ]:
# datos del futuro
dfuture <- dataset[foto_mes == 202109]

In [ ]:
# scoring, aplico el modelo a los datos del futuro
prediccion <- predict(modelo_pruned,
  dfuture,
  type= "prob"
)

In [ ]:
# tabla prediccion
tb_prediccion <- as.data.table(list(
  "numero_de_cliente" = dfuture$numero_de_cliente,
  "prob"=prediccion[, "POS"]
))

In [ ]:
# Decison
setorder( tb_prediccion, -prob )
tb_prediccion[ , Predicted := 0L ]
tb_prediccion[ seq(PARAM$envios), Predicted := 1L ]


In [ ]:
# archivo para kaggle
archivo_kaggle <- "KBin5500_005.csv"

fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

In [ ]:
# subida a Kaggle
comando <- "kaggle competitions submit"
competencia <- "-c data-mining-inicial-2026-b"
arch <- paste( "-f", archivo_kaggle)

In [ ]:
mensaje <- paste0("-m 'cp=", PARAM$rpart$cp,
  "  minsplit=", PARAM$rpart$minsplit,
  "  minbucket=", PARAM$rpart$minbucket,
  "  maxdepth=", PARAM$rpart$maxdepth,
  "  envios=", PARAM$envios,
  "  peso=", PARAM$peso,
  "'"
 )

In [ ]:
linea <- paste( comando, competencia, arch, mensaje)

# este es el comando que correria desde el prompt de Linux
linea

In [ ]:
# ejecuto el comando
salida <- system(linea, intern=TRUE)
cat(salida)

Los resultados quedan en el repo local: **`exp/exp5500/`**
<br> abra el archivo **stopping_at_canaritos.pdf**
<br> abra el .pdf con un lector de PDF




---

